# 02 - Pilot 20K (verify VRAM/settings)

Train on 20K samples with full config (3 epochs). Verify VRAM fits, callback runs at least once,
and MAE preview. Do NOT push to Hub. ETA: ~35-45 min on RTX 5090.

**Acceptance:** no OOM, callback runs >= 1 time, final MAE < 200K VND.

In [ ]:
# Cell 1 - Imports + constants
import os, sys, torch, wandb
sys.path.insert(0, os.path.abspath("."))

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

from utils.items_vn import load_items, DATASET_NAME
from utils.training_utils import (
    get_bnb_config, get_lora_config, get_sft_config,
    DataCollatorCompletionOnly, MaeEvalCallback,
    MAX_SEQ_LENGTH, VAL_EVAL_SIZE, RESPONSE_TEMPLATE,
)

BASE_MODEL    = "Qwen/Qwen3.5-4B-Base"
OUTPUT_DIR    = "outputs/qwen_v2_pilot"
HUB_MODEL_ID  = None
PILOT_SIZE    = 20_000

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"CUDA: {torch.cuda.is_available()}")
print(f"GPU:  {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Cell 2 - Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

template_ids = tokenizer.encode(RESPONSE_TEMPLATE, add_special_tokens=False)
print(f"RESPONSE_TEMPLATE ids: {template_ids}")
print(f"RESPONSE_TEMPLATE decoded: '{tokenizer.decode(template_ids)}'")

In [ ]:
# Cell 3 - Load + format + tokenize pilot subset (20K)
def format_and_tokenize(example):
    text = example["prompt"] + example["completion"] + tokenizer.eos_token
    return tokenizer(
        text,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding=False,
    )

print(f"Loading {PILOT_SIZE:,} train rows...")
raw_train = load_dataset(DATASET_NAME, split="train").select(range(PILOT_SIZE))
print(f"  Raw train: {len(raw_train):,} rows")

train_ds = raw_train.map(
    format_and_tokenize,
    batched=True,
    remove_columns=raw_train.column_names,
    desc="Tokenising pilot",
)
print(f"  Tokenised train: {len(train_ds):,} rows")

In [ ]:
# Cell 4 - Load model (4-bit)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=get_bnb_config(),
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False

print(f"Model loaded. VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# Cell 5 - Apply LoRA
lora_cfg = get_lora_config()
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

In [ ]:
# Cell 6 - Load val items for MAE callback
print(f"Loading {VAL_EVAL_SIZE} val items for MAE callback...")
val_items = load_items("validation", size=VAL_EVAL_SIZE)
print(f"  Loaded: {len(val_items)} items")

In [ ]:
# Cell 7 - Build callback + collator + trainer
mae_callback = MaeEvalCallback(
    model=model,
    tokenizer=tokenizer,
    val_items=val_items,
    output_dir=OUTPUT_DIR,
)
collator = DataCollatorCompletionOnly(tokenizer)
sft_cfg  = get_sft_config(OUTPUT_DIR, HUB_MODEL_ID)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_ds,
    data_collator=collator,
    args=sft_cfg,
    callbacks=[mae_callback],
)
print("Trainer ready.")

In [ ]:
# Cell 8 - W&B init + train pilot
wandb.init(project="qwen-vn-pricer-v2", name="v2-pilot-20k")

trainer.train()

print("\n=== MAE History (pilot) ===")
for step, mae in mae_callback.history:
    marker = " <-- BEST" if step == mae_callback.best_step else ""
    print(f"  step={step:>5}  mae={mae:.2f}K VND{marker}")
print(f"\nBest pilot ckpt at step {mae_callback.best_step}: MAE={mae_callback.best_mae:.2f}K VND")

In [ ]:
# Cell 9 - Save pilot results JSON. Do NOT push to Hub.
import json
results = {
    "size": PILOT_SIZE,
    "best_mae_k": round(mae_callback.best_mae, 2),
    "best_step": mae_callback.best_step,
    "history": [[s, round(m, 2)] for s, m in mae_callback.history],
}
os.makedirs("results", exist_ok=True)
with open("results/pilot_20k_results.json", "w") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print(json.dumps(results, indent=2))
wandb.finish()